<a href="https://colab.research.google.com/github/SanjaraT/Langchain/blob/main/chromaDB_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain-huggingface
!pip install sentence-transformers
!pip install chromadb
!pip install langchain-chroma

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

In [ ]:
# Open-source embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [7]:
# documents
doc1 = Document(
    page_content="Pizza is a popular Italian dish made with dough, tomato sauce, cheese, and various toppings.",
    metadata={"category": "Italian"}
)

doc2 = Document(
    page_content="Biryani is a flavorful rice dish cooked with spices, meat, or vegetables and is popular across South Asia.",
    metadata={"category": "South Asian"}
)

doc3 = Document(
    page_content="Sushi is a Japanese dish consisting of vinegared rice combined with seafood, vegetables, or egg.",
    metadata={"category": "Japanese"}
)

doc4 = Document(
    page_content="Burger is a sandwich made with a ground meat patty, vegetables, and sauces inside a bun.",
    metadata={"category": "Fast Food"}
)

doc5 = Document(
    page_content="Pad Thai is a famous Thai noodle dish made with rice noodles, peanuts, vegetables, and protein.",
    metadata={"category": "Thai"}
)

In [8]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [9]:
# Create Chroma DB
vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory="food_chroma_db",
    collection_name="foods"
)

In [10]:
# Add documents
vector_store.add_documents(docs)

['8140f5d2-3060-4fc4-9b20-fe03e61ca243',
 '88ef1a53-f2f0-4bb1-acf2-5611b2d20854',
 'a1946444-cce9-4000-a7af-a5161dee15d6',
 '3ea6b33e-be0b-4ce9-8adf-da4563d40996',
 'edaaa783-c114-418d-8f82-0c6a6b5bbd13']

In [11]:
results = vector_store.similarity_search(
    query="Which food comes from Japan?",
    k=2
)

for doc in results:
    print(doc.page_content)

Sushi is a Japanese dish consisting of vinegared rice combined with seafood, vegetables, or egg.
Pad Thai is a famous Thai noodle dish made with rice noodles, peanuts, vegetables, and protein.


In [12]:
results = vector_store.similarity_search_with_score(
    query="Which food comes from Japan?",
    k=2
)

for doc, score in results:
    print(score)
    print(doc.page_content)
    print()

1.0282431840896606
Sushi is a Japanese dish consisting of vinegared rice combined with seafood, vegetables, or egg.

1.1933238506317139
Pad Thai is a famous Thai noodle dish made with rice noodles, peanuts, vegetables, and protein.



In [13]:
vector_store.similarity_search(
    query="",
    filter={"category": "Japanese"}
)

[Document(id='a1946444-cce9-4000-a7af-a5161dee15d6', metadata={'category': 'Japanese'}, page_content='Sushi is a Japanese dish consisting of vinegared rice combined with seafood, vegetables, or egg.')]

In [18]:
# Update
updated_doc = Document(
    page_content="Sushi is a traditional Japanese dish made with vinegared rice and seafood.",
    metadata={"category": "Japanese"},
    id="a1946444-cce9-4000-a7af-a5161dee15d6"
)

vector_store.update_document(
    document_id="a1946444-cce9-4000-a7af-a5161dee15d6",
    document=updated_doc
)

In [19]:
print(vector_store.get(ids=["a1946444-cce9-4000-a7af-a5161dee15d6"]))

{'ids': ['a1946444-cce9-4000-a7af-a5161dee15d6'], 'embeddings': None, 'documents': ['Sushi is a traditional Japanese dish made with vinegared rice and seafood.'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'category': 'Japanese'}]}


In [20]:
# Delete
vector_store.delete(ids=["a1946444-cce9-4000-a7af-a5161dee15d6"])

In [21]:
print(vector_store.get())

{'ids': ['8140f5d2-3060-4fc4-9b20-fe03e61ca243', '88ef1a53-f2f0-4bb1-acf2-5611b2d20854', '3ea6b33e-be0b-4ce9-8adf-da4563d40996', 'edaaa783-c114-418d-8f82-0c6a6b5bbd13'], 'embeddings': None, 'documents': ['Pizza is a popular Italian dish made with dough, tomato sauce, cheese, and various toppings.', 'Biryani is a flavorful rice dish cooked with spices, meat, or vegetables and is popular across South Asia.', 'Burger is a sandwich made with a ground meat patty, vegetables, and sauces inside a bun.', 'Pad Thai is a famous Thai noodle dish made with rice noodles, peanuts, vegetables, and protein.'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'category': 'Italian'}, {'category': 'South Asian'}, {'category': 'Fast Food'}, {'category': 'Thai'}]}
